### Evaluate on myself-annotations

In [12]:
import sys
import json
from types import SimpleNamespace

import numpy as np
import torch
from sklearn.metrics import roc_curve, roc_auc_score

sys.path.append("../")
from clip import clip
from src.utils.dataset import AugmentedDataset

In [13]:
id2label_cub = {0: 'black footed albatross', 1: 'laysan albatross', 2: 'sooty albatross', 3: 'groove billed ani', 4: 'crested auklet', 5: 'least auklet', 6: 'parakeet auklet', 7: 'rhinoceros auklet', 8: 'brewer blackbird', 9: 'red winged blackbird', 10: 'rusty blackbird', 11: 'yellow headed blackbird', 12: 'bobolink', 13: 'indigo bunting', 14: 'lazuli bunting', 15: 'painted bunting', 16: 'cardinal', 17: 'spotted catbird', 18: 'gray catbird', 19: 'yellow breasted chat', 20: 'eastern towhee', 21: 'chuck will widow', 22: 'brandt cormorant', 23: 'red faced cormorant', 24: 'pelagic cormorant', 25: 'bronzed cowbird', 26: 'shiny cowbird', 27: 'brown creeper', 28: 'american crow', 29: 'fish crow', 30: 'black billed cuckoo', 31: 'mangrove cuckoo', 32: 'yellow billed cuckoo', 33: 'gray crowned rosy finch', 34: 'purple finch', 35: 'northern flicker', 36: 'acadian flycatcher', 37: 'great crested flycatcher', 38: 'least flycatcher', 39: 'olive sided flycatcher', 40: 'scissor tailed flycatcher', 41: 'vermilion flycatcher', 42: 'yellow bellied flycatcher', 43: 'frigatebird', 44: 'northern fulmar', 45: 'gadwall', 46: 'american goldfinch', 47: 'european goldfinch', 48: 'boat tailed grackle', 49: 'eared grebe', 50: 'horned grebe', 51: 'pied billed grebe', 52: 'western grebe', 53: 'blue grosbeak', 54: 'evening grosbeak', 55: 'pine grosbeak', 56: 'rose breasted grosbeak', 57: 'pigeon guillemot', 58: 'california gull', 59: 'glaucous winged gull', 60: 'heermann gull', 61: 'herring gull', 62: 'ivory gull', 63: 'ring billed gull', 64: 'slaty backed gull', 65: 'western gull', 66: 'anna hummingbird', 67: 'ruby throated hummingbird', 68: 'rufous hummingbird', 69: 'green violetear', 70: 'long tailed jaeger', 71: 'pomarine jaeger', 72: 'blue jay', 73: 'florida jay', 74: 'green jay', 75: 'dark eyed junco', 76: 'tropical kingbird', 77: 'gray kingbird', 78: 'belted kingfisher', 79: 'green kingfisher', 80: 'pied kingfisher', 81: 'ringed kingfisher', 82: 'white breasted kingfisher', 83: 'red legged kittiwake', 84: 'horned lark', 85: 'pacific loon', 86: 'mallard', 87: 'western meadowlark', 88: 'hooded merganser', 89: 'red breasted merganser', 90: 'mockingbird', 91: 'nighthawk', 92: 'clark nutcracker', 93: 'white breasted nuthatch', 94: 'baltimore oriole', 95: 'hooded oriole', 96: 'orchard oriole', 97: 'scott oriole', 98: 'ovenbird', 99: 'brown pelican', 100: 'white pelican', 101: 'western wood pewee', 102: 'sayornis', 103: 'american pipit', 104: 'whip poor will', 105: 'horned puffin', 106: 'common raven', 107: 'white necked raven', 108: 'american redstart', 109: 'geococcyx', 110: 'loggerhead shrike', 111: 'great grey shrike', 112: 'baird sparrow', 113: 'black throated sparrow', 114: 'brewer sparrow', 115: 'chipping sparrow', 116: 'clay colored sparrow', 117: 'house sparrow', 118: 'field sparrow', 119: 'fox sparrow', 120: 'grasshopper sparrow', 121: 'harris sparrow', 122: 'henslow sparrow', 123: 'le conte sparrow', 124: 'lincoln sparrow', 125: 'nelson sharp tailed sparrow', 126: 'savannah sparrow', 127: 'seaside sparrow', 128: 'song sparrow', 129: 'tree sparrow', 130: 'vesper sparrow', 131: 'white crowned sparrow', 132: 'white throated sparrow', 133: 'cape glossy starling', 134: 'bank swallow', 135: 'barn swallow', 136: 'cliff swallow', 137: 'tree swallow', 138: 'scarlet tanager', 139: 'summer tanager', 140: 'artic tern', 141: 'black tern', 142: 'caspian tern', 143: 'common tern', 144: 'elegant tern', 145: 'forsters tern', 146: 'least tern', 147: 'green tailed towhee', 148: 'brown thrasher', 149: 'sage thrasher', 150: 'black capped vireo', 151: 'blue headed vireo', 152: 'philadelphia vireo', 153: 'red eyed vireo', 154: 'warbling vireo', 155: 'white eyed vireo', 156: 'yellow throated vireo', 157: 'bay breasted warbler', 158: 'black and white warbler', 159: 'black throated blue warbler', 160: 'blue winged warbler', 161: 'canada warbler', 162: 'cape may warbler', 163: 'cerulean warbler', 164: 'chestnut sided warbler', 165: 'golden winged warbler', 166: 'hooded warbler', 167: 'kentucky warbler', 168: 'magnolia warbler', 169: 'mourning warbler', 170: 'myrtle warbler', 171: 'nashville warbler', 172: 'orange crowned warbler', 173: 'palm warbler', 174: 'pine warbler', 175: 'prairie warbler', 176: 'prothonotary warbler', 177: 'swainson warbler', 178: 'tennessee warbler', 179: 'wilson warbler', 180: 'worm eating warbler', 181: 'yellow warbler', 182: 'northern waterthrush', 183: 'louisiana waterthrush', 184: 'bohemian waxwing', 185: 'cedar waxwing', 186: 'american three toed woodpecker', 187: 'pileated woodpecker', 188: 'red bellied woodpecker', 189: 'red cockaded woodpecker', 190: 'red headed woodpecker', 191: 'downy woodpecker', 192: 'bewick wren', 193: 'cactus wren', 194: 'carolina wren', 195: 'house wren', 196: 'marsh wren', 197: 'rock wren', 198: 'winter wren', 199: 'common yellowthroat'}
label2id_cub = {'black footed albatross': 0, 'laysan albatross': 1, 'sooty albatross': 2, 'groove billed ani': 3, 'crested auklet': 4, 'least auklet': 5, 'parakeet auklet': 6, 'rhinoceros auklet': 7, 'brewer blackbird': 8, 'red winged blackbird': 9, 'rusty blackbird': 10, 'yellow headed blackbird': 11, 'bobolink': 12, 'indigo bunting': 13, 'lazuli bunting': 14, 'painted bunting': 15, 'cardinal': 16, 'spotted catbird': 17, 'gray catbird': 18, 'yellow breasted chat': 19, 'eastern towhee': 20, 'chuck will widow': 21, 'brandt cormorant': 22, 'red faced cormorant': 23, 'pelagic cormorant': 24, 'bronzed cowbird': 25, 'shiny cowbird': 26, 'brown creeper': 27, 'american crow': 28, 'fish crow': 29, 'black billed cuckoo': 30, 'mangrove cuckoo': 31, 'yellow billed cuckoo': 32, 'gray crowned rosy finch': 33, 'purple finch': 34, 'northern flicker': 35, 'acadian flycatcher': 36, 'great crested flycatcher': 37, 'least flycatcher': 38, 'olive sided flycatcher': 39, 'scissor tailed flycatcher': 40, 'vermilion flycatcher': 41, 'yellow bellied flycatcher': 42, 'frigatebird': 43, 'northern fulmar': 44, 'gadwall': 45, 'american goldfinch': 46, 'european goldfinch': 47, 'boat tailed grackle': 48, 'eared grebe': 49, 'horned grebe': 50, 'pied billed grebe': 51, 'western grebe': 52, 'blue grosbeak': 53, 'evening grosbeak': 54, 'pine grosbeak': 55, 'rose breasted grosbeak': 56, 'pigeon guillemot': 57, 'california gull': 58, 'glaucous winged gull': 59, 'heermann gull': 60, 'herring gull': 61, 'ivory gull': 62, 'ring billed gull': 63, 'slaty backed gull': 64, 'western gull': 65, 'anna hummingbird': 66, 'ruby throated hummingbird': 67, 'rufous hummingbird': 68, 'green violetear': 69, 'long tailed jaeger': 70, 'pomarine jaeger': 71, 'blue jay': 72, 'florida jay': 73, 'green jay': 74, 'dark eyed junco': 75, 'tropical kingbird': 76, 'gray kingbird': 77, 'belted kingfisher': 78, 'green kingfisher': 79, 'pied kingfisher': 80, 'ringed kingfisher': 81, 'white breasted kingfisher': 82, 'red legged kittiwake': 83, 'horned lark': 84, 'pacific loon': 85, 'mallard': 86, 'western meadowlark': 87, 'hooded merganser': 88, 'red breasted merganser': 89, 'mockingbird': 90, 'nighthawk': 91, 'clark nutcracker': 92, 'white breasted nuthatch': 93, 'baltimore oriole': 94, 'hooded oriole': 95, 'orchard oriole': 96, 'scott oriole': 97, 'ovenbird': 98, 'brown pelican': 99, 'white pelican': 100, 'western wood pewee': 101, 'sayornis': 102, 'american pipit': 103, 'whip poor will': 104, 'horned puffin': 105, 'common raven': 106, 'white necked raven': 107, 'american redstart': 108, 'geococcyx': 109, 'loggerhead shrike': 110, 'great grey shrike': 111, 'baird sparrow': 112, 'black throated sparrow': 113, 'brewer sparrow': 114, 'chipping sparrow': 115, 'clay colored sparrow': 116, 'house sparrow': 117, 'field sparrow': 118, 'fox sparrow': 119, 'grasshopper sparrow': 120, 'harris sparrow': 121, 'henslow sparrow': 122, 'le conte sparrow': 123, 'lincoln sparrow': 124, 'nelson sharp tailed sparrow': 125, 'savannah sparrow': 126, 'seaside sparrow': 127, 'song sparrow': 128, 'tree sparrow': 129, 'vesper sparrow': 130, 'white crowned sparrow': 131, 'white throated sparrow': 132, 'cape glossy starling': 133, 'bank swallow': 134, 'barn swallow': 135, 'cliff swallow': 136, 'tree swallow': 137, 'scarlet tanager': 138, 'summer tanager': 139, 'artic tern': 140, 'black tern': 141, 'caspian tern': 142, 'common tern': 143, 'elegant tern': 144, 'forsters tern': 145, 'least tern': 146, 'green tailed towhee': 147, 'brown thrasher': 148, 'sage thrasher': 149, 'black capped vireo': 150, 'blue headed vireo': 151, 'philadelphia vireo': 152, 'red eyed vireo': 153, 'warbling vireo': 154, 'white eyed vireo': 155, 'yellow throated vireo': 156, 'bay breasted warbler': 157, 'black and white warbler': 158, 'black throated blue warbler': 159, 'blue winged warbler': 160, 'canada warbler': 161, 'cape may warbler': 162, 'cerulean warbler': 163, 'chestnut sided warbler': 164, 'golden winged warbler': 165, 'hooded warbler': 166, 'kentucky warbler': 167, 'magnolia warbler': 168, 'mourning warbler': 169, 'myrtle warbler': 170, 'nashville warbler': 171, 'orange crowned warbler': 172, 'palm warbler': 173, 'pine warbler': 174, 'prairie warbler': 175, 'prothonotary warbler': 176, 'swainson warbler': 177, 'tennessee warbler': 178, 'wilson warbler': 179, 'worm eating warbler': 180, 'yellow warbler': 181, 'northern waterthrush': 182, 'louisiana waterthrush': 183, 'bohemian waxwing': 184, 'cedar waxwing': 185, 'american three toed woodpecker': 186, 'pileated woodpecker': 187, 'red bellied woodpecker': 188, 'red cockaded woodpecker': 189, 'red headed woodpecker': 190, 'downy woodpecker': 191, 'bewick wren': 192, 'cactus wren': 193, 'carolina wren': 194, 'house wren': 195, 'marsh wren': 196, 'rock wren': 197, 'winter wren': 198, 'common yellowthroat': 199}
classnames_cub = ['black footed albatross', 'laysan albatross', 'sooty albatross', 'groove billed ani', 'crested auklet', 'least auklet', 'parakeet auklet', 'rhinoceros auklet', 'brewer blackbird', 'red winged blackbird', 'rusty blackbird', 'yellow headed blackbird', 'bobolink', 'indigo bunting', 'lazuli bunting', 'painted bunting', 'cardinal', 'spotted catbird', 'gray catbird', 'yellow breasted chat', 'eastern towhee', 'chuck will widow', 'brandt cormorant', 'red faced cormorant', 'pelagic cormorant', 'bronzed cowbird', 'shiny cowbird', 'brown creeper', 'american crow', 'fish crow', 'black billed cuckoo', 'mangrove cuckoo', 'yellow billed cuckoo', 'gray crowned rosy finch', 'purple finch', 'northern flicker', 'acadian flycatcher', 'great crested flycatcher', 'least flycatcher', 'olive sided flycatcher', 'scissor tailed flycatcher', 'vermilion flycatcher', 'yellow bellied flycatcher', 'frigatebird', 'northern fulmar', 'gadwall', 'american goldfinch', 'european goldfinch', 'boat tailed grackle', 'eared grebe', 'horned grebe', 'pied billed grebe', 'western grebe', 'blue grosbeak', 'evening grosbeak', 'pine grosbeak', 'rose breasted grosbeak', 'pigeon guillemot', 'california gull', 'glaucous winged gull', 'heermann gull', 'herring gull', 'ivory gull', 'ring billed gull', 'slaty backed gull', 'western gull', 'anna hummingbird', 'ruby throated hummingbird', 'rufous hummingbird', 'green violetear', 'long tailed jaeger', 'pomarine jaeger', 'blue jay', 'florida jay', 'green jay', 'dark eyed junco', 'tropical kingbird', 'gray kingbird', 'belted kingfisher', 'green kingfisher', 'pied kingfisher', 'ringed kingfisher', 'white breasted kingfisher', 'red legged kittiwake', 'horned lark', 'pacific loon', 'mallard', 'western meadowlark', 'hooded merganser', 'red breasted merganser', 'mockingbird', 'nighthawk', 'clark nutcracker', 'white breasted nuthatch', 'baltimore oriole', 'hooded oriole', 'orchard oriole', 'scott oriole', 'ovenbird', 'brown pelican', 'white pelican', 'western wood pewee', 'sayornis', 'american pipit', 'whip poor will', 'horned puffin', 'common raven', 'white necked raven', 'american redstart', 'geococcyx', 'loggerhead shrike', 'great grey shrike', 'baird sparrow', 'black throated sparrow', 'brewer sparrow', 'chipping sparrow', 'clay colored sparrow', 'house sparrow', 'field sparrow', 'fox sparrow', 'grasshopper sparrow', 'harris sparrow', 'henslow sparrow', 'le conte sparrow', 'lincoln sparrow', 'nelson sharp tailed sparrow', 'savannah sparrow', 'seaside sparrow', 'song sparrow', 'tree sparrow', 'vesper sparrow', 'white crowned sparrow', 'white throated sparrow', 'cape glossy starling', 'bank swallow', 'barn swallow', 'cliff swallow', 'tree swallow', 'scarlet tanager', 'summer tanager', 'artic tern', 'black tern', 'caspian tern', 'common tern', 'elegant tern', 'forsters tern', 'least tern', 'green tailed towhee', 'brown thrasher', 'sage thrasher', 'black capped vireo', 'blue headed vireo', 'philadelphia vireo', 'red eyed vireo', 'warbling vireo', 'white eyed vireo', 'yellow throated vireo', 'bay breasted warbler', 'black and white warbler', 'black throated blue warbler', 'blue winged warbler', 'canada warbler', 'cape may warbler', 'cerulean warbler', 'chestnut sided warbler', 'golden winged warbler', 'hooded warbler', 'kentucky warbler', 'magnolia warbler', 'mourning warbler', 'myrtle warbler', 'nashville warbler', 'orange crowned warbler', 'palm warbler', 'pine warbler', 'prairie warbler', 'prothonotary warbler', 'swainson warbler', 'tennessee warbler', 'wilson warbler', 'worm eating warbler', 'yellow warbler', 'northern waterthrush', 'louisiana waterthrush', 'bohemian waxwing', 'cedar waxwing', 'american three toed woodpecker', 'pileated woodpecker', 'red bellied woodpecker', 'red cockaded woodpecker', 'red headed woodpecker', 'downy woodpecker', 'bewick wren', 'cactus wren', 'carolina wren', 'house wren', 'marsh wren', 'rock wren', 'winter wren', 'common yellowthroat']

In [14]:
args = SimpleNamespace(
    image_dir = "../cub_visible_part_bench/images"
)
args.checkpoint_path = "../checkpoints_acc/cub_auroc_soft_01_aug_color02_qwen3-4b/ctr_nonvis_rich2_noclass_l2sp0.1_lamda3.0_ep12/run_1/best.pt"
args.checkpoint_path = "../checkpoints_acc/cub_auroc_soft_01_aug_color02_qwen3-4b/ctr_nonvis_rich2_noclass_l2sp0.5_lamda3.0_ep10/run_1/best.pt"
args.checkpoint_path = "../checkpoints_acc/cub_auroc_soft_01_aug_color02_qwen3-4b/ctr_nonvis_rich2_noclass_l2sp0.5_lamda3.0_ep10_hard/run_1/best.pt"
args.checkpoint_path = "../checkpoints_acc/cub_auroc_soft_01_aug_color02_qwen3-4b/ctr_nonvis_rich2_noclass_l2sp0.5_lamda3.0_ep10_hard0.3/run_1/best.pt"
args.checkpoint_path = "../checkpoints_acc/cub_auroc_soft_01_aug_color02_qwen3-4b/ctr_nonvis_rich2_noclass_l2sp0.5_lamda3.0_ep10_hard0.6/run_1/best.pt"
args.checkpoint_path = "../checkpoints_acc/cub_auroc_soft_01_aug_color02_qwen3-4b/ctr_nonvis_rich2_noclass_l2sp0.5_lamda3.0_ep10_hard0.5_bi/run_1/best.pt"
args.checkpoint_path = "../checkpoints_acc/cub_auroc_soft_01_aug_color02_qwen3-4b/ctr_nonvis_rich2_noclass_l2sp0.5_lamda3.0_ep10_no_aug/run_1/best.pt"

args.checkpoint_path = '../checkpoints_acc4/cub_visible_all_vit16/lambda_2b/ours_lambda6.0_ep1_v4/run_/best.pt'



In [15]:
# %%
device = "cuda" if torch.cuda.is_available() else "cpu"
# model, preprocess = clip.load("ViT-B/32", device=device, jit=False)
model, preprocess = clip.load("ViT-B/16", device=device, jit=False)
checkpoint = torch.load(args.checkpoint_path)
model.load_state_dict(checkpoint['model_state_dict'])
print(f'load... {args.checkpoint_path}')


/tmp/ipykernel_29750/2221824420.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(args.checkpoint_path)


load... ../checkpoints_acc4/cub_visible_all_vit16/lambda_2b/ours_lambda6.0_ep1_v4/run_/best.pt


In [16]:
dataset = AugmentedDataset(args.image_dir, transform=preprocess)

In [17]:
with open("../descriptors/my_cub2-1.json", "r") as f:
    descriptors = json.load(f)

In [18]:
# %%
##concept計算
concept_ids_train = []
for classname in classnames_cub:
    class_concept = descriptors[classname]
    join_concept = ", ".join(class_concept[:-1]) + " and " + class_concept[-1]
    class_concept = [f"{classname} with {x}." for x in class_concept]
    class_concept = [join_concept] + class_concept
    concept_ids_train.append(clip.tokenize(class_concept).unsqueeze(0))
concept_ids_train = torch.concat(concept_ids_train)
text_ids = concept_ids_train
class_concept

['olive-brown head with black mask, dark eyes, thin pointed beak, yellow throat patch neck, bright yellow breast, pale yellow belly, olive-brown back, olive-brown wings, pinkish legs and short olive-brown tail',
 'common yellowthroat with olive-brown head with black mask.',
 'common yellowthroat with dark eyes.',
 'common yellowthroat with thin pointed beak.',
 'common yellowthroat with yellow throat patch neck.',
 'common yellowthroat with bright yellow breast.',
 'common yellowthroat with pale yellow belly.',
 'common yellowthroat with olive-brown back.',
 'common yellowthroat with olive-brown wings.',
 'common yellowthroat with pinkish legs.',
 'common yellowthroat with short olive-brown tail.']

In [19]:
model.eval()
all_text_features = []
for i in range(text_ids.size(0)):
    text_features = model.encode_text(text_ids[i].to(device))
    text_features = text_features.detach().cpu()
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)
    all_text_features.append(text_features.unsqueeze(0))
text_features = torch.concat(all_text_features)
text_features.shape


torch.Size([200, 11, 512])

In [20]:
with open('../cub_visible_part_bench/anno_myself_class50_100.json', 'r') as f:
    anno = json.load(f)

count = {'head': 0, 'eye': 0, 'beak': 0, 'neck': 0, 'breast': 0, 'belly': 0, 'back': 0, 'wing': 0, 'leg': 0, 'tail': 0}
for x in anno:
    visible_labels = anno[x]
    for part in count:
        count[part] +=visible_labels[part]
len(anno), count


(100,
 {'head': 50,
  'eye': 44,
  'beak': 53,
  'neck': 62.5,
  'breast': 54.5,
  'belly': 49.5,
  'back': 42.0,
  'wing': 71.0,
  'leg': 40.5,
  'tail': 47.5})

In [21]:
def compute_fpr95(labels, scores):
    """
    labels: (N,) binary {0,1}  1 = positive (visible)
    scores: (N,) confidence scores (higher = more positive)
    """
    fpr, tpr, _ = roc_curve(labels, scores)

    # TPR >= 0.95 となる最小の FPR
    if np.max(tpr) < 0.95:
        return np.nan

    idx = np.where(tpr >= 0.95)[0][0]
    return fpr[idx]


def compute_imagewise_auroc_fpr95(dataset, model, text_features, anno, device):
    model.eval()
    aurocs = []
    fpr95s = []
    scores_all = []

    with torch.no_grad():
        for sample in dataset:
                pixel_values = sample[0][None, :].to(device)
                class_id = sample[1]
                filename = sample[2]
                visible_mask = list(anno[filename].values())
                visible_mask = torch.tensor(visible_mask)

                image_features = model.encode_image(pixel_values)
                image_features = image_features / image_features.norm(dim=-1, keepdim=True)
                scores = (
                    image_features.cpu()
                    @ text_features[class_id, 1:].T
                ).numpy()[0]

                labels = visible_mask.cpu().numpy()
                mask = (labels != 0.5)
                labels_bin = labels[mask].astype(np.int32)
                scores = scores[mask]

                scores_all.append((scores, labels_bin))
                aurocs.append(
                    roc_auc_score(labels_bin, scores)
                )
                
                fpr95 = compute_fpr95(labels_bin, scores)
                if not np.isnan(fpr95):
                    fpr95s.append(fpr95)
        print(f'concept lack image number: {len(aurocs)}')
        print(f"AUROC: {float(np.mean(aurocs))}")
        print(f"FPR96: {float(np.mean(fpr95s))}")
        return scores_all
                

In [22]:
print(args.checkpoint_path)
scores_all = compute_imagewise_auroc_fpr95(dataset, model, text_features, anno, device)

../checkpoints_acc4/cub_visible_all_vit16/lambda_2b/ours_lambda6.0_ep1_v4/run_/best.pt
concept lack image number: 100
AUROC: 0.728607738095238
FPR96: 0.5260873015873015
